# 01 · Data: labels, cleaning, split by car model

**Stanford Cars**: 16,185 photos of 196 car models (make, model, year). Each model name contains its body type (e.g. *BMW X5 **SUV** 2007*), so we turn the 196 models into two labels:

- **big** = SUV, pickup truck (*Cab*), van, minivan
- **small** = sedan, hatchback, coupe, convertible, wagon

Then we clean the data and split it **by car family**, so the test set only has car models the network never saw.

The first run downloads ~2 GB (about 3–5 min) and saves a resized copy to Drive so the next notebooks start fast.

In [ ]:
# --- Setup: same cell in every notebook ---
# Colab: the repo lives in Google Drive (shared by all 4 notebooks); images go on the fast local disk.
import os, sys
REPO_URL = "https://github.com/YUPWE/car-size.git"   # your repo
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = "/content/drive/MyDrive/car-size"
    if not os.path.exists(ROOT):
        !git clone {REPO_URL} {ROOT}
    os.chdir(ROOT)
    !pip install -q -r requirements.txt
    os.environ["CARS_DATA_DIR"] = "/content/cars_data"
else:
    ROOT = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
    os.chdir(ROOT)
sys.path.insert(0, ROOT)
os.makedirs("results", exist_ok=True)
os.makedirs("weights", exist_ok=True)
QUICK = os.environ.get("QUICK", "0") == "1"   # tiny run, only used to test that the code works
print("Working in", os.getcwd(), "| QUICK =", QUICK)

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from PIL import Image
from src import data

data.download()
df = data.load_meta()
print(len(df), "images,", df.model.nunique(), "car models,", df.group.nunique(), "car families")
print(df.label.value_counts())

fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
df.body.value_counts().plot.bar(ax=ax[0], title="Images per body type",
    color=["tab:red" if b in data.BIG_BODY else "tab:blue" for b in df.body.value_counts().index])
df.label.value_counts().plot.bar(ax=ax[1], title="Images per label", color=["tab:blue", "tab:red"])
plt.tight_layout(); plt.savefig("results/class_counts.png", dpi=120); plt.show()

In [ ]:
# How each of the 196 model names was labelled (15 were labelled by hand: no body-type word in the name)
models = df.drop_duplicates("model")[["model", "body", "label"]].sort_values("model")
print(models[models.model.isin(data.MANUAL_BODY)].to_string(index=False))
models.to_csv("results/model_labels.csv", index=False)

In [ ]:
fig, axes = plt.subplots(2, 6, figsize=(18, 6))
for r, lab in enumerate(data.CLASSES):
    for k, (_, row) in enumerate(df[df.label == lab].sample(6, random_state=1).iterrows()):
        axes[r, k].imshow(Image.open(f"{data.IMG_DIR}/{row.path}")); axes[r, k].axis("off")
        axes[r, k].set_title(f"{lab}: {row.model[:28]}", fontsize=8)
plt.tight_layout(); plt.savefig("results/examples.png", dpi=100); plt.show()

## Cleaning
**1. Tiny images.** Some originals are thumbnails (shorter side under 64 px): too little detail to trust the label.

In [ ]:
short = df[["orig_w", "orig_h"]].min(axis=1)
short.plot.hist(bins=50, range=(0, 1200), figsize=(6, 3), title="Shorter side of the original image (px)"); plt.show()
tiny = df.index[short < 64]
print(len(tiny), "images with shorter side < 64 px;", (short < 128).sum(), "under 128 px")

**2. Near-duplicates across car families.** A perceptual hash turns each image into 64 bits; near-identical photos differ in ≤ 4 bits. The same photo filed under two *different* families is a label error and could leak between train and test.

In [ ]:
hashes = data.perceptual_hashes(df.path)
pairs = data.near_duplicate_pairs(hashes, max_dist=4)
cross = [(i, j) for i, j in pairs if df.group[i] != df.group[j]]
print(len(pairs), "near-duplicate pairs;", len(cross), "of them across different car families")

show = cross[:6]
fig, axes = plt.subplots(len(show), 2, figsize=(8, 2.8 * len(show)), squeeze=False)
for r, (i, j) in enumerate(show):
    for k, idx in enumerate([i, j]):
        axes[r, k].imshow(Image.open(f"{data.IMG_DIR}/{df.path[idx]}")); axes[r, k].axis("off")
        axes[r, k].set_title(df.model[idx], fontsize=9)
plt.tight_layout(); plt.savefig("results/duplicates_across_models.png", dpi=100); plt.show()

In [ ]:
drop = sorted(set(tiny) | {k for p in cross for k in p})
clean = df.drop(index=drop).reset_index(drop=True)
print(f"Dropped {len(drop)} images -> {len(clean)} remain")

## Split by car family
All years and variants of a family (e.g. every *Chevrolet Silverado*) go to one split. So the test score tells us how the model does on **car models it has never seen** — the situation in a real car park.

In [ ]:
split_df = data.make_splits(clean, seed=42)
print(pd.crosstab(split_df.label, split_df.split)[["train", "val", "test"]])
print(split_df.split.value_counts(normalize=True).round(3))
print("\nbig share per split:", split_df.groupby("split").label.apply(lambda s: round((s == "big").mean(), 3)).to_dict())

# Leakage checks
assert (split_df.groupby("group").split.nunique() == 1).all(), "a car family is in two splits!"
assert (split_df.groupby("model").split.nunique() == 1).all()
print("OK: no car family appears in two splits")
print("\nTest car families:", sorted(split_df[split_df.split == "test"].group.unique()))

In [ ]:
data.save_splits(split_df)
print(os.listdir("splits"))

### Known problems found
- Big/small comes from the **body type in the name**, not measured length: a long luxury sedan counts as *small*, a compact SUV as *big*. Wagons are labelled small.
- About 1/3 big vs 2/3 small → class-weighted loss, and we report precision/recall for *big*.
- Photos come from the web (ads, reviews, car shows): many clean side/front views, unlike a parking-entrance camera.
- Tiny images and cross-model duplicate photos (removed above).